# EEG analysis assistant (Claude) in Google Colab

Talk to an AI assistant about your EEG recording. It asks what kind of study you are doing and
what you want to find out, proposes a processing plan, runs each step with the eegproc toolkit,
shows you the figures, and asks for your view before every decision that changes the data.

**You need an Anthropic API key** (from console.anthropic.com). API use is paid per token; the
session prints how many tokens it used when you type `quit`.

**What leaves this notebook:** summaries, tables and figures produced by the tools go to the
Anthropic API. The raw signals do not, and patient fields and the recording date from file
headers are never included. This is a research and teaching tool, not a medical device.

## 1. Install

Takes about 20 seconds; no restart needed.

In [ ]:
!pip install -q "eegproc[assistant] @ git+https://github.com/rsd21/EEg-singnal-processing-.git"

## 2. Add your API key

Click the key icon (**Secrets**) in Colab's left sidebar, add a secret named
`ANTHROPIC_API_KEY` with your key as the value, and switch on **Notebook access**. Then run:

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("API key loaded")
except ImportError:
    print("Not in Colab: set ANTHROPIC_API_KEY in your environment instead")

## 3. Choose a recording

Leave `UPLOAD = False` to start with a simulated 2-minute recording with eye blinks. Set it to
`True` to upload your own file: EDF, BDF, BrainVision (select the `.vhdr`, `.vmrk` and `.eeg`
files together), CSV/TXT, MATLAB `.mat`, EEGLAB `.set`, HDF5 or XDF.

In [ ]:
UPLOAD = False
recording = "demo:blinks"

if UPLOAD:
    from google.colab import files
    uploaded = list(files.upload())
    recording = next((f for f in uploaded if f.lower().endswith((".vhdr", ".set"))), uploaded[0])
print("Recording:", recording)

## 4. Talk to the assistant

Type your answers in the box that appears under the cell and press Enter. Type `quit` to finish.
Figures appear inline and are saved, together with any data or reports, in the
`eeg_assistant_output` folder (Files panel on the left).

In [ ]:
from eegproc.assistant import chat

assistant = chat(recording)

## 5. Download the results

In [ ]:
import shutil

archive = shutil.make_archive("eeg_assistant_output", "zip", "eeg_assistant_output")
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Saved", archive)